In [1]:
import os
import pandas as pd
import numpy as np

# Navigate up from notebooks/ into data/
RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)

def standardize_text_columns(df):
    """Trims leading/trailing whitespace and collapses internal spaces."""
    for col in df.select_dtypes(include=['object', 'string']).columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.replace(r'\s+', ' ', regex=True)
        )
    return df

print("=== Starting Step 1: Data Cleaning ===")

# 1. Main Dataset: IDS_ALLCountries_Data.csv
main_path = os.path.join(RAW_DIR, "IDS_ALLCountries_Data.csv")
print(f"Loading {main_path}...")
df_all = pd.read_csv(main_path, encoding='latin1', low_memory=False)

# Drop 5 empty/footer metadata rows where Country Code is NaN
df_all = df_all.dropna(subset=['Country Code']).copy()

# Drop sparse forecast columns (2025 to 2032 >= 60% nulls)
col_null_pct = (df_all.isnull().sum() / len(df_all)) * 100
cols_to_drop = col_null_pct[col_null_pct >= 60].index.tolist()
print(f"Dropping columns with >= 60% nulls: {cols_to_drop}")
df_all.drop(columns=cols_to_drop, inplace=True)

# Standardize text
df_all = standardize_text_columns(df_all)

# Year columns to numeric
year_cols = [c for c in df_all.columns if c.isdigit()]
df_all[year_cols] = df_all[year_cols].apply(pd.to_numeric, errors='coerce')

# Non-zero imputation pipeline
print("Applying non-zero imputation (ffill/bfill + indicator median)...")
df_all[year_cols] = df_all[year_cols].ffill(axis=1).bfill(axis=1)

for yr in year_cols:
    df_all[yr] = df_all.groupby('Series Code')[yr].transform(lambda s: s.fillna(s.median()))

for yr in year_cols:
    df_all[yr] = df_all[yr].fillna(df_all[yr].median())

# Save cleaned wide version
wide_out = os.path.join(PROCESSED_DIR, "cleaned_IDS_ALLCountries_Data_wide.csv")
df_all.to_csv(wide_out, index=False)
print(f"Saved wide dataset: {wide_out} (Shape: {df_all.shape})")

# Unpivot into normalized long format (Fact Table)
print("Converting to normalized long format...")
id_vars = [
    'Country Name', 'Country Code', 
    'Counterpart-Area Name', 'Counterpart-Area Code', 
    'Series Name', 'Series Code'
]
df_long = pd.melt(
    df_all,
    id_vars=id_vars,
    value_vars=year_cols,
    var_name='Year',
    value_name='Debt_Value'
)
df_long['Year'] = df_long['Year'].astype(int)
df_long['Debt_Value'] = df_long['Debt_Value'].astype(float)

long_out = os.path.join(PROCESSED_DIR, "cleaned_IDS_ALLCountries_Data_long.csv")
df_long.to_csv(long_out, index=False)
print(f"Saved long dataset: {long_out} (Shape: {df_long.shape})")

# 2. Country Metadata: IDS_CountryMetaData.csv
c_path = os.path.join(RAW_DIR, "IDS_CountryMetaData.csv")
print(f"\nLoading {c_path}...")
df_country = pd.read_csv(c_path, encoding='latin1')
df_country.drop_duplicates(inplace=True)

cols_drop_c = df_country.columns[(df_country.isnull().sum() / len(df_country)) * 100 >= 60].tolist()
print(f"Dropping Country Meta columns (>=60% null): {cols_drop_c}")
df_country.drop(columns=cols_drop_c, inplace=True)
df_country = standardize_text_columns(df_country)

for col in df_country.select_dtypes(include=['object', 'string']).columns:
    df_country[col] = df_country[col].replace({'nan': 'Unknown', 'None': 'Unknown'})

c_out = os.path.join(PROCESSED_DIR, "cleaned_IDS_CountryMetaData.csv")
df_country.to_csv(c_out, index=False)
print(f"Saved: {c_out} (Shape: {df_country.shape})")

# 3. Series Metadata: IDS_SeriesMetaData.csv
s_path = os.path.join(RAW_DIR, "IDS_SeriesMetaData.csv")
print(f"\nLoading {s_path}...")
df_series = pd.read_csv(s_path, encoding='latin1')
df_series.drop_duplicates(inplace=True)

cols_drop_s = df_series.columns[(df_series.isnull().sum() / len(df_series)) * 100 >= 60].tolist()
print(f"Dropping Series Meta columns (>=60% null): {cols_drop_s}")
df_series.drop(columns=cols_drop_s, inplace=True)
df_series = standardize_text_columns(df_series)

for col in df_series.select_dtypes(include=['object', 'string']).columns:
    df_series[col] = df_series[col].replace({'nan': 'Not Specified', 'None': 'Not Specified'})

s_out = os.path.join(PROCESSED_DIR, "cleaned_IDS_SeriesMetaData.csv")
df_series.to_csv(s_out, index=False)
print(f"Saved: {s_out} (Shape: {df_series.shape})")

# 4. Linkage Tables
fn_path = os.path.join(RAW_DIR, "IDS_FootNoteMetaData.csv")
df_fn = pd.read_csv(fn_path, encoding='latin1')
df_fn.drop_duplicates(inplace=True)
df_fn = standardize_text_columns(df_fn)
df_fn.to_csv(os.path.join(PROCESSED_DIR, "cleaned_IDS_FootNoteMetaData.csv"), index=False)

cs_path = os.path.join(RAW_DIR, "Country-Series - Metadata.csv")
df_cs = pd.read_csv(cs_path, encoding='latin1')
df_cs.drop_duplicates(inplace=True)
df_cs = standardize_text_columns(df_cs)
df_cs.to_csv(os.path.join(PROCESSED_DIR, "cleaned_Country_Series_Metadata.csv"), index=False)

print("\n=== Step 1 Finished: All datasets cleaned! ===")

=== Starting Step 1: Data Cleaning ===
Loading ../data/raw/IDS_ALLCountries_Data.csv...
Dropping columns with >= 60% nulls: ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032']
Applying non-zero imputation (ffill/bfill + indicator median)...
Saved wide dataset: ../data/processed/cleaned_IDS_ALLCountries_Data_wide.csv (Shape: (62978, 31))
Converting to normalized long format...
Saved long dataset: ../data/processed/cleaned_IDS_ALLCountries_Data_long.csv (Shape: (1574450, 8))

Loading ../data/raw/IDS_CountryMetaData.csv...
Dropping Country Meta columns (>=60% null): ['Other groups', 'National accounts reference year', 'Vital registration complete']
Saved: ../data/processed/cleaned_IDS_CountryMetaData.csv (Shape: (134, 27))

Loading ../data/raw/IDS_SeriesMetaData.csv...
Dropping Series Meta columns (>=60% null): ['License Type', 'Limitations and exceptions', 'General comments']
Saved: ../data/processed/cleaned_IDS_SeriesMetaData.csv (Shape: (574, 9))

=== Step 1 Finished: All 